# Testing the `populartimes` Package

Fetches popular times data for restaurants near **Times Square, NYC** using the [populartimes](https://github.com/m-wrzr/populartimes) library.

**Requirements:**
- A Google Places API key (with Places API enabled)
- `populartimes` package (`pip install git+https://github.com/m-wrzr/populartimes.git`)

In [ ]:
# Install populartimes (uncomment if needed)
# !pip install --use-pep517 git+https://github.com/m-wrzr/populartimes.git

In [ ]:
import populartimes
import json
import pandas as pd
from google.colab import userdata

# Get API key — uses Colab secrets; fall back to manual entry
try:
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    API_KEY = input("Enter your Google Places API key: ")

## Method 1: Search by area using `populartimes.get()`

Search for restaurants within a bounding box around Times Square.

- **SW corner:** ~40.7560, -73.9895
- **NE corner:** ~40.7600, -73.9840

In [ ]:
# Bounding box around Times Square
p1 = (40.7560, -73.9895)  # SW corner
p2 = (40.7600, -73.9840)  # NE corner

# Fetch restaurants in the area
results = populartimes.get(
    API_KEY,
    ["restaurant"],
    p1,
    p2,
    radius=100,
    all_places=False  # only places that have populartimes data
)

print(f"Found {len(results)} restaurants with popular times data")

In [ ]:
# Show the first result as a sample
if results:
    print(json.dumps(results[0], indent=2))

In [ ]:
# Build a summary DataFrame
rows = []
for place in results:
    rows.append({
        "name": place.get("name"),
        "place_id": place.get("id"),
        "address": place.get("address"),
        "rating": place.get("rating"),
        "rating_n": place.get("rating_n"),
        "lat": place.get("coordinates", {}).get("lat"),
        "lng": place.get("coordinates", {}).get("lng"),
        "has_populartimes": place.get("populartimes") is not None,
        "current_popularity": place.get("current_popularity"),
    })

df = pd.DataFrame(rows)
print(f"{len(df)} restaurants found")
df.head(20)

## Method 2: Look up a specific place using `populartimes.get_id()`

Fetch detailed popular times for a single place by its Google Place ID.

In [ ]:
# Example: use the first place_id from our search, or a known Times Square restaurant
if results:
    place_id = results[0]["id"]
else:
    # Fallback: Olive Garden Times Square
    place_id = "ChIJRYBMj_BYwokRzKFaj7CUVwM"

place_detail = populartimes.get_id(API_KEY, place_id)
print(f"Place: {place_detail['name']}")
print(f"Address: {place_detail['address']}")
print(f"Current popularity: {place_detail.get('current_popularity', 'N/A')}")

In [ ]:
# Parse popular times into a DataFrame
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pt_rows = []

if "populartimes" in place_detail and place_detail["populartimes"]:
    for day_data in place_detail["populartimes"]:
        day_name = day_data["name"]
        for hour, popularity in enumerate(day_data["data"]):
            pt_rows.append({
                "day": day_name,
                "hour": hour,
                "popularity": popularity
            })

df_pt = pd.DataFrame(pt_rows)
print(f"Popular times data for: {place_detail['name']}")
df_pt.head(10)

In [ ]:
# Pivot to show popularity by day and hour
if not df_pt.empty:
    pivot = df_pt.pivot(index="hour", columns="day", values="popularity")
    pivot = pivot[days]  # reorder columns
    pivot

In [ ]:
# Simple visualization
if not df_pt.empty:
    pivot.plot(figsize=(14, 6), title=f"Popular Times — {place_detail['name']}")

## Explore wait times and time spent

The `populartimes` package can also return:
- `time_spent`: typical time spent at the location (min, max in minutes)
- `time_wait`: estimated wait times by day/hour

In [ ]:
print(f"Time spent: {place_detail.get('time_spent', 'N/A')}")

if "time_wait" in place_detail and place_detail["time_wait"]:
    wait_rows = []
    for day_data in place_detail["time_wait"]:
        day_name = day_data["name"]
        for hour, wait in enumerate(day_data["data"]):
            wait_rows.append({"day": day_name, "hour": hour, "wait_minutes": wait})
    df_wait = pd.DataFrame(wait_rows)
    print("\nWait times sample:")
    display(df_wait[df_wait["wait_minutes"] > 0].head(10))
else:
    print("No wait time data available for this place.")